# LMECA2300: Advanced Numerical Methods
## Assignment 3

**Students:**
- Student 1: Victor Lepère (61502000)
- Student 2: Max Opdecam(77062000)

Let us consider the shape $\Gamma$ as a collection of $N$ segments $\Gamma_i$. We will denote their midpoint by $\mathbf m_i$. At the middle of each segment, we would like to impose the total pressure to be zero, i.e. 
\begin{align*}
0 = P_0 e^{-j k \hat{\mathbf{u}} \cdot \mathbf{m}_j}  + \frac{\omega \rho_0}{4} \int_\Gamma H_0^{(2)} (k||\mathbf m_j - \mathbf r' ||_2) \, v_n(\mathbf r') \, d\Gamma' \qquad j = 1, \dotsc, N
\end{align*}
Where  $\hat{\mathbf{u}}$ is the unit direction of the incident field of pressure. We now decompose the unknowns velocities as linear combination of piecewise unit-constant basis functions whose support restricts to a single segment as follows
$$
v_n = \sum_{i=1}^N x_i v_{n,i} \quad \text{ with } v_{n,i} = \begin{cases}
1 \text{ on } \Gamma_i \\
0 \text{ otherwise}
\end{cases} 
$$
Substituing back into the previous expression, we get:
\begin{align*}
0 = P_0 e^{-j k \hat{\mathbf{u}} \cdot \mathbf{m}_j}  + \frac{\omega \rho_0}{4} &\int_\Gamma H_0^{(2)} (k||\mathbf m_j - \mathbf r' ||_2) \, \sum_{i=1}^N x_i v_{n,i} \, d\Gamma' \qquad &j = 1, \dotsc, N \\
\sum_{i=1}^N x_i &\int_\Gamma H_0^{(2)} (k||\mathbf m_j - \mathbf r' ||_2) \, v_{n,i} \, d\Gamma' = -\frac{4 P_0}{\omega \rho_0} e^{-j k \hat{\mathbf{u}} \cdot \mathbf{m}_j} \qquad &j = 1, \dotsc, N \\
\sum_{i=1}^N x_i & \underbrace{\int_{\Gamma_i} H_0^{(2)} (k||\mathbf m_j - \mathbf r' ||_2) \, d\Gamma'}_{A_{j i}} = \underbrace{-\frac{4 P_0}{\omega \rho_0} e^{-j k \hat{\mathbf{u}} \cdot \mathbf{m}_j}}_{b_j} \qquad &j = 1, \dotsc, N
\end{align*}
Hence we end up with a linear system $A x = b$. Solving it yields the distribution of normal velocities over the segments. Based on this distribution, we can compute the total pressure everywhere using techniques of the previous assignments. See the implementation below.

In [ ]:
### Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
from scipy.special import hankel2
from math import ceil

### Physical constants
rho0 = 1.2 # air, 20°C, 1 atm
f = 1/16 # [hz]
w = 2*np.pi*f # air -> 343 = w/k
k = w/343
lambda_ = (2*np.pi)/k

### A segment
class Edge:
    def __init__(self,r1,r2):
        self.r1 = r1
        self.r2 = r2

    def __integrate_hankel(self, a, b, m, case):
        """
        Aux. function that integrates Hankel function from a to b

        @param a: lower intergration bound
        @param b: upper integration bound
        @param case: symmetric/general case
        @param m: number of integration points
        """
        s = np.linspace(a, b, m)
        if case == 0:
            diff = self.obs - (self.r1 + s[:, np.newaxis] * (self.obs - self.r1))
        elif case == 1:
            diff = s[:, np.newaxis] * (self.r2 - self.obs)
        else:
            diff = self.obs - (self.r1 + s[:, np.newaxis] * (self.r2 - self.r1))
        rho = np.linalg.norm(diff, axis=1)
        singularity_mask = k*rho <= lambda_/1000

        evals = np.zeros_like(rho, dtype=np.complex128)
        cst = 1 - 1j*(2/np.pi)*(np.euler_gamma - np.log(2))
        evals[singularity_mask] = cst
        subset_rho = rho[~singularity_mask]
        evals[~singularity_mask] = hankel2(0, k*subset_rho) + 1j*(2/np.pi)*np.log(k*subset_rho)

        h = (b - a) / m
        evals[0] *= 1/2
        evals[-1] *= 1/2
        first_term = h*np.sum(evals)

        if case == 0:
            if b == 1:
                last = 0
            else:
                last = (1-b)*(np.log(1-b)-1)
            second_term = -1j*(2/np.pi)*((b-a)*(np.log(k)+np.log(np.linalg.norm(self.obs-self.r1))) + ((1-a)*(np.log(1-a)-1) - last))
        elif case == 1:
            if a == 0:
                last = 0
            else:
                last = a*(np.log(a)-1)
            second_term = -1j*(2/np.pi)*((b-a)*(np.log(k)+np.log(np.linalg.norm(self.r2-self.obs))) + (b*(np.log(b)-1) - last))
        else:
            pass

        return first_term + second_term
    
    def integrate_sym(self, Vn, obs): # TO FIX !!!!
        """
        Integrates complex amplitude of radiated pressure wave for an obsever point ON the segment, and thus takes advantage of symmetry.

        @param Vn: constant normal velocity
        @param obs: coord. of observer
        """
        self.obs = obs
        L1 = np.linalg.norm(obs - self.r1)
        L2 = np.linalg.norm(obs - self.r2)

        sbar = np.fmin(L1, L2) / np.fmax(L1, L2)

        m = 20
        if L1 == 0:
            return (Vn*w*rho0*L2/4)*self.__integrate_hankel(0, 1, m, 1)
        elif L2 == 0:
            return (Vn*w*rho0*L1/4)*self.__integrate_hankel(0, 1, m, 0)
        elif L1 > L2:
            return (Vn*w*rho0/4)*(2*L2*self.__integrate_hankel(0, 1-sbar, m, 0) + L2*self.__integrate_hankel(1-sbar, 1, m, 0))
        elif L1 < L2:
            return (Vn*w*rho0/4)*(L1*self.__integrate_hankel(0, sbar, m, 1) + 2*L1*self.__integrate_hankel(sbar, 1, m, 1))
        else:
            return (Vn*w*rho0*L1/2)*self.__integrate_hankel(0, 1, m, 0)
        
    def integrate_gen(self, Vn, obs):
        """
        Integrates complex amplitude of radiated pressure wave fo an observer point 
        """